# Raaga.AI × ACE-Step 1.5: raga-locked songs with realistic vocals

Runtime → Change runtime type → **T4 GPU** (or better).

1. Raaga.AI composes raga-valid svaras and Telugu lyrics, then renders a **guide track**.
2. **ACE-Step 1.5** (MIT license) re-performs the guide in `cover` mode with realistic vocals and instruments, singing the lyrics.
3. Every candidate is pitch-tracked and scored for **raga adherence**. The best one is saved as `song.wav`.

The first run downloads about 10 GB of ACE-Step weights to Drive. Later runs reuse them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/raagaai'
!mkdir -p {DRIVE_DIR}/songs {DRIVE_DIR}/ace_checkpoints

In [ ]:
%cd /content
!git clone -q https://github.com/Pariseashishbannu/raagaai.git
!git clone -q --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
# Keep model weights on Drive so they survive runtime resets
!rm -rf /content/ACE-Step-1.5/checkpoints && ln -s {DRIVE_DIR}/ace_checkpoints /content/ACE-Step-1.5/checkpoints
!pip install -q -e /content/ACE-Step-1.5
!pip install -q numpy scipy soundfile structlog pydantic-settings librosa
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Generate
- `STRENGTH`: higher keeps the melody closer to the guide (more raga-accurate); lower sounds more natural. Try 0.6–0.85.
- `CANDIDATES`: more candidates give a better chance of a high raga score, but each one adds render time.
- Turn on `OFFLOAD` if you hit CUDA out-of-memory errors on a T4.

In [ ]:
RAGA = 'mohanam'          # any id in data/ragas, e.g. kalyani, shankarabharanam, hamsadhwani
TALA = 'adi'
KALA = 'madhyama'         # vilamba | madhyama | druta
THEME = 'devotional'      # devotional | romantic | nature | celebration | melancholy
VOICE = 'female'          # female | male
STYLE = ''                # extra style words, e.g. 'Telugu film song, lush strings'
STRENGTH = 0.7
CANDIDATES = 4
OFFLOAD = False

import time
OUT = f'{DRIVE_DIR}/songs/{RAGA}_{int(time.time())}'
%cd /content/raagaai
!python -m scripts.hybrid_song --raga {RAGA} --tala {TALA} --kala {KALA} \
    --theme {THEME} --voice {VOICE} --style "{STYLE}" \
    --strength {STRENGTH} --candidates {CANDIDATES} {'--offload' if OFFLOAD else ''} \
    --ace-root /content/ACE-Step-1.5 --out-dir {OUT}

## Listen

In [ ]:
import json
from IPython.display import Audio, display
meta = json.load(open(f'{OUT}/song.json'))
print(meta['title'], '| raga score', meta['raga_score'], '| guide', meta['guide_raga_score'])
print(meta['lyrics'])
print('Guide track (Raaga.AI synth):'); display(Audio(f'{OUT}/guide.wav'))
print('Final song (ACE-Step):');        display(Audio(f'{OUT}/song.wav'))